In [ ]:
# 0,1: 00 for psi_1, 01 for psi_2 ...
# 2,3: For states
# 4,5: Ancillary qubits

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from numpy import sqrt, array


def circuit_init():
    qc = QuantumCircuit(6)
    return qc
    
def PREP(qc):

    desired_vector = [
        1/sqrt(3), 0, 0, 0, 
        1/(4*sqrt(3)), 1/4, 1/4, sqrt(3)/4, 
        1/(4*sqrt(3)), -1/4, -1/4, sqrt(3)/4, 
        0, 0, 0, 0
    ]
    qc.initialize(desired_vector, [3,2,1,0])
    return qc

def Unitary(qc, wires, num_layers):

    x = ParameterVector('x', 4 * num_layers)
    z = ParameterVector('z', 4 * num_layers)
    
    for i in range(num_layers):
        qc.barrier()
        qc.rx(x[4*i], wires[0])
        qc.rx(x[4*i+1], wires[1])
        qc.rx(x[4*i+2], wires[2])
        qc.rx(x[4*i+3], wires[3])

        qc.rz(z[4*i], wires[0])
        qc.rz(z[4*i+1], wires[1])
        qc.rz(z[4*i+2], wires[2])
        qc.rz(z[4*i+3], wires[3])
        
        # Apply CNOT gates between the qubits
        qc.cx(wires[0], wires[1])
        qc.cx(wires[1], wires[2]) 
        qc.cx(wires[2], wires[3])
        qc.cx(wires[3], wires[0])
    
    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_layers=3
num_params= 8*num_layers
parmas_instant = rand(num_params)
num_shots=10000

qc = circuit_init()
qc = PREP(qc)
Unitary(qc, [2,3,4,5] , num_layers)

qc.draw(output='mpl', style = 'clifford') 
qc_reversed=qc.reverse_bits()

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel='ibm_quantum')
#backend = service.least_busy(min_num_qubits=127)
backend = service.backend("ibm_strasbourg")
print(backend)

pm = generate_preset_pass_manager(optimization_level=3,backend=backend)


candidate_circuit = pm.run(qc_reversed)
candidate_circuit.draw('mpl', fold=False, idle_wires=False)

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Define the factor (I + Z)/2 = |0><0|
o_term = SparsePauliOp.from_list([("I", 0.5), ("Z", 0.5)])

# Define the factor (I - Z)/2 = |1><1|
l_term = SparsePauliOp.from_list([("I", 0.5), ("Z", -0.5)])

I = SparsePauliOp.from_list([("I", 1)])

# Use tensor products to create (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2
Ob1= o_term.tensor(o_term).tensor(I).tensor(I).tensor(o_term).tensor(o_term)
Ob2= o_term.tensor(l_term).tensor(I).tensor(I).tensor(o_term).tensor(l_term)
Ob3= l_term.tensor(o_term).tensor(I).tensor(I).tensor(l_term).tensor(o_term)

cost_hamiltonian = Ob1 + Ob2 + Ob3
cost_hamiltonian=cost_hamiltonian.apply_layout(candidate_circuit.layout)

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # Prepare the job input with the ansatz, Hamiltonian, and parameters
    pub = (ansatz, hamiltonian, params)
    job = estimator.run([pub])

    # Retrieve the result
    results = job.result()[0]

    # Extract the cost (expectation value)
    cost = results.data.evs
    print('cost is', cost)

    # Append cost to the global success probability list and return it
    success_probability.append(cost)
    return 1/cost

In [ ]:
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

init_params = rand(num_params)
success_probability = [] # Global variable

with Session(backend=backend) as session:

    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000
    #estimator.options.max_execution_time = 4500

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-1,
    )
    print(result)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(success_probability, label="Success Probability")
plt.xlabel('Iteration')
plt.ylabel('Sucess Probability')
plt.legend()
plt.show()

In [ ]:
success_probability = [0.22318692230203369
, 0.25358667305873744
, 0.29398769252172496
, 0.30204024120633954
, 0.25534503375282663
, 0.22846563233894293
, 0.25526674168236546
, 0.1803049306395783
, 0.25680241539983173
, 0.2591987413488212
, 0.2479576481513312
, 0.2682529944803663
, 0.25353989953206546
, 0.26029073997341307
, 0.274306109841005
, 0.2206007747285559
, 0.3099558534634612
, 0.25564120950712627
, 0.2842684266521666
, 0.28671156284076094
, 0.26230635522344725
, 0.27651571104260586
, 0.26068615558287883
, 0.2806414540330649
, 0.2681155909590788
, 0.30483587571476284
, 0.26977243156054
, 0.28447543792729063
, 0.29069536523963263
, 0.3075034004693628
, 0.28783011996754265
, 0.2672120586293549
, 0.3070531786354271
, 0.27703818148206816
, 0.2956864102435409
, 0.24545142687258542
, 0.29465499323742905
, 0.3001251790014165
, 0.29750883055498095
, 0.2978668059106604
, 0.2492210071403969
, 0.2511930390655904
, 0.2890453361011986
, 0.29658590788264005
, 0.2971488540229913
, 0.2922624433850933
, 0.2795221404328251
, 0.2748476074536606
, 0.2840292345300528
, 0.2964743410905992
, 0.2784318142609981
, 0.27046707065221565
, 0.3052618946067853
, 0.27678951112203964
, 0.3016596837632764
, 0.2660207085557944
, 0.26862678239098725
, 0.27148859325879443
, 0.28772086608827485
, 0.2840021046123154
, 0.28989782039061773
, 0.30420114393748465
, 0.2802499777926025
, 0.27151734521056514
, 0.2966897260513257
, 0.28840170432768464
, 0.3006628982201007
, 0.2818606116093741
, 0.27854226176372665
, 0.2912070852995087
, 0.3067188219086523
, 0.30173235154199
, 0.28768805930297914]

In [ ]:
import openpyxl

# create a new workbook
workbook = openpyxl.Workbook()

# select the active worksheet
worksheet = workbook.active

# loop through the confidences array and write values to the worksheet
for i in range(len(success_probability)):
    worksheet.cell(row=i+1, column=1, value=float(success_probability[i]))

# save the workbook to a file
workbook.save('ME HEA(C) DT states tol01 (ibm_strasbourg).xlsx')